# `scheme_management` 02: noteworthy single-feature findings

**Purpose:** identify and discuss supported points that stand out after the
standard `category` breakdown. Target relationships here are
exploratory and must be rechecked after the split is frozen.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'scheme_management'
feature_metadata = {'order': 20, 'name': 'scheme_management', 'audit_type': 'category', 'role': 'candidate', 'disposition': 'retain and ablate against management', 'finding': 'The field has source blanks and strongly overlaps management without being equivalent.', 'decision': 'Keep blank explicit and compare its incremental value with management.', 'risk': 'Parallel management fields may mostly add redundancy.', 'sentinel_tokens': ['None'], 'related': [{'feature': 'management', 'reason': 'Both describe management using overlapping labels.'}, {'feature': 'scheme_name', 'reason': 'Named schemes have imperfect management mappings.'}, {'feature': 'management_group', 'reason': 'This is the broader management concept.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for scheme_management.


## Supported target evidence


In [2]:
sentinel_tokens = ['None']
target_profile = categorical_target_profile(
    training_data,
    feature,
    minimum_support=100,
    sentinel_tokens=sentinel_tokens,
)
display(target_profile.head(20))

supported = target_profile.loc[target_profile["meets support threshold"]].copy()
non_functional_column = "non functional (%)"
if non_functional_column in supported:
    display(
        supported.sort_values(non_functional_column, ascending=False)
        .head(12)[["rows", non_functional_column]]
    )


status_group,rows,meets support threshold,functional (%),functional needs repair (%),non functional (%)
scheme_management,,,,,
vwc,36793,True,51.53,6.34,42.12
wug,5206,True,57.74,12.91,29.35
<missing/blank>,3877,True,48.31,5.75,45.94
water authority,3153,True,51.32,14.21,34.48
wua,2883,True,69.20,8.29,22.51
water board,2748,True,74.71,4.04,21.25
parastatal,1680,True,57.50,12.02,30.48
private operator,1063,True,68.58,2.16,29.26
company,1061,True,50.33,3.49,46.18


status_group,rows,non functional (%)
scheme_management,,
company,1061,46.18
<missing/blank>,3877,45.94
vwc,36793,42.12
other,766,37.34
water authority,3153,34.48
parastatal,1680,30.48
wug,5206,29.35
private operator,1063,29.26
wua,2883,22.51


## Observation

The field has source blanks and strongly overlaps management without being equivalent.

## Interpretation

The supported single-feature patterns make this field worth the stated
treatment, but they do not prove causation or independent predictive value.
High-cardinality and geographic fields are especially vulnerable to
memorisation under a random split.

## Provisional decision

Keep blank explicit and compare its incremental value with management.

**Risk to carry forward:** Parallel management fields may mostly add redundancy.


In [3]:
decision_record = pd.DataFrame([{
    "feature": feature,
    "role": feature_metadata["role"],
    "disposition": feature_metadata["disposition"],
    "finding": feature_metadata["finding"],
    "decision": feature_metadata["decision"],
    "risk": feature_metadata["risk"],
}])
display(decision_record.set_index("feature"))


,role,disposition,finding,decision,risk
feature,,,,,
scheme_management,candidate,retain and ablate against management,The field has source blanks and strongly overl...,Keep blank explicit and compare its incrementa...,Parallel management fields may mostly add redu...
